In [0]:
-- =============================================================================
-- SLEEPER CORE PIPELINE
-- Dynasty-Ready Analytics Foundation
-- 
-- Key Features:
-- - Player dimension populated from Sleeper players API
-- - Weekly player performance with position and starter status
-- - League configuration for dynasty/keeper/redraft classification
-- - Complete player ownership lifecycle tracking
-- - Enhanced waiver and draft analytics
-- =============================================================================

In [0]:
-- ---------- DIMENSIONS ----------


In [0]:
-- =============================================================================
-- dim_league_clusters: League groups tracked across multiple seasons
-- =============================================================================
-- PURPOSE:
--   Groups leagues by name to enable cross-season dynasty analysis. Sleeper creates
--   new league_ids each season, so we use a normalized cluster_key to track the same
--   league across years.
--
-- GRAIN: 
--   One row per unique league (identified by cluster_key)
--
-- COLUMNS:
--   cluster_key:    Normalized league identifier for cross-season tracking (URL-safe)
--   cluster_name:   Human-readable league name
--   season_start:   First season this league appears in data
--   season_end:     Most recent season this league appears in data
--
-- DEPENDENCIES:
--   - sleeper_league_info_snapshot: Raw league metadata from Sleeper API
-- =============================================================================
CREATE OR REPLACE MATERIALIZED VIEW dim_league_clusters AS
SELECT
  lower(regexp_replace(coalesce(name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  any_value(name) AS cluster_name,
  min(season) AS season_start,
  max(season) AS season_end
FROM workspace.sleeper_raw.sleeper_league_info_snapshot
GROUP BY lower(regexp_replace(coalesce(name, 'unknown'), '[^a-zA-Z0-9]+', '_'));

In [0]:
-- =============================================================================
-- dim_manager_roster_map: Manager identity mapping across seasons
-- =============================================================================
-- PURPOSE:
--   Maps roster_ids to manager identities with username alias reconciliation.
--   Critical for tracking manager performance across seasons and handling 
--   username changes (e.g., TakeFlightJetUp -> akumthekar).
--
-- GRAIN:
--   One row per (league_id, season, roster_id) combination
--
-- COLUMNS:
--   league_id:              Sleeper's league identifier (changes each season)
--   season:                 Fantasy season year (YYYY)
--   roster_id:              Team identifier within a specific league/season
--   manager_user_id:        Sleeper user ID (stable across username changes)
--   manager_display_name:   Canonical manager username (with aliases resolved)
--   cluster_key:            League cluster for cross-season tracking
--   cluster_name:           Human-readable league name
--
-- BUSINESS LOGIC:
--   - Aliases CTE: Manual mapping for username changes (add new entries as needed)
--   - Uses coalesce(alias, raw_name) to prefer canonical names over raw usernames
--   - LEFT JOINs preserve rosters even if manager/league data is missing
--
-- DEPENDENCIES:
--   - sleeper_rosters_snapshot: Raw roster data
--   - sleeper_users_snapshot: Manager/user data
--   - sleeper_league_info_snapshot: League metadata
-- =============================================================================
CREATE OR REPLACE MATERIALIZED VIEW dim_manager_roster_map AS
WITH aliases AS (
  -- Manager alias mapping for username changes
  -- Add new rows here when managers change their display names
  SELECT 'TakeFlightJetUp' AS old_name, 'akumthekar' AS canonical_name
),
ro AS (
  SELECT league_id, roster_id, owner_id
  FROM workspace.sleeper_raw.sleeper_rosters_snapshot
),
us AS (
  SELECT league_id AS u_league_id, user_id AS manager_user_id, display_name
  FROM workspace.sleeper_raw.sleeper_users_snapshot
),
li AS (
  SELECT league_id, season, name 
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
),
base_map AS (
  SELECT
    ro.league_id,
    li.season,
    ro.roster_id,
    us.manager_user_id,
    coalesce(us.display_name, 'Unknown') AS raw_display_name,
    lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
    li.name AS cluster_name
  FROM ro
  LEFT JOIN us ON ro.league_id = us.u_league_id AND ro.owner_id = us.manager_user_id
  LEFT JOIN li ON ro.league_id = li.league_id
)
SELECT
  bm.league_id,
  bm.season,
  bm.roster_id,
  bm.manager_user_id,
  coalesce(a.canonical_name, bm.raw_display_name) AS manager_display_name,
  bm.cluster_key,
  bm.cluster_name
FROM base_map bm
LEFT JOIN aliases a ON bm.raw_display_name = a.old_name;

In [0]:
-- =============================================================================
-- dim_player_ownership: Complete player ownership lifecycle tracking
-- =============================================================================
-- PURPOSE:
--   Tracks every ownership period for every player on every roster. Critical for
--   dynasty analysis - answers "when did I own player X" and "is this player
--   currently on my roster". 
--
-- GRAIN:
--   One row per ownership period (acquisition + optional departure)
--
-- COLUMNS:
--   ownership_id:             Unique identifier for this ownership period (MD5 hash)
--   player_id:                Sleeper player identifier
--   league_id:                Sleeper league identifier for the acquisition season
--   season:                   Season when player was acquired
--   roster_id:                Roster that owns/owned the player
--   acquired_date:            Timestamp when player was acquired (NULL for drafts)
--   acquired_week:            Week number when acquired (1 for drafts)
--   acquired_via:             How player was acquired (startup_draft, rookie_draft, trade, waiver, free_agent)
--   acquired_transaction_id:  Transaction or draft ID for acquisition event
--   acquired_faab:            FAAB spent on waiver acquisition (0 for other types)
--   departed_date:            Timestamp when player left roster (NULL if still owned)
--   departed_week:            Week number when departed (NULL if still owned)
--   departed_via:             How player left (trade, drop, NULL if still owned)
--   departed_transaction_id:  Transaction ID for departure event (NULL if still owned)
--   is_current_roster:        TRUE if player is currently on this roster
--   weeks_owned:              Number of weeks player was owned (NULL if cross-season)
--
-- KEY BUSINESS LOGIC:
--   1. Extracts ALL acquisition events (drafts, trades, waivers) with cluster_key
--   2. Extracts ALL departure events (trades, drops) with cluster_key
--   3. Matches each acquisition to its NEXT departure using:
--      - Same player_id
--      - Same cluster_key (enables cross-season matching for dynasty leagues)
--      - Same roster_id
--      - Departure timestamp > Acquisition timestamp
--   4. Uses ROW_NUMBER to pick the earliest matching departure per acquisition
--   5. Records with departed_date=NULL represent current ownership
--
-- IMPORTANT NOTES:
--   - Uses cluster_key instead of league_id for cross-season matching (Sleeper
--     creates new league_ids each season)
--   - Filters to status='complete' transactions only (prevents failed waiver
--     claims from creating ownership records)
--   - No season constraint on departure matching (allows cross-season ownership)
--
-- DATA QUALITY:
--   - is_current_roster should NEVER be TRUE if departed_date is not NULL
--   - weeks_owned can be NULL for cross-season ownership (calculation is complex)
--   - Multiple ownership periods for same player/roster indicate boomerang player
--
-- DEPENDENCIES:
--   - sleeper_draft_picks_snapshot: Draft acquisition events
--   - sleeper_drafts_snapshot: Draft metadata
--   - sleeper_transactions_snapshot: Trade/waiver/drop events  
--   - sleeper_league_info_snapshot: League metadata for cluster_key
-- =============================================================================
CREATE OR REPLACE MATERIALIZED VIEW dim_player_ownership AS
WITH 
-- Step 1: Extract all acquisition events with cluster_key
draft_acquisitions AS (
  SELECT
    dp.player_id,
    dp.league_id,
    dr.season,
    dp.roster_id,
    CAST(NULL AS TIMESTAMP) AS acquired_date,  -- Drafts don't have exact timestamps
    1 AS acquired_week,
    CASE 
      WHEN dr.season = (
        SELECT min(season) 
        FROM workspace.sleeper_raw.sleeper_league_info_snapshot 
        WHERE league_id = dp.league_id
      )
      THEN 'startup_draft'  -- First season = startup draft
      ELSE 'rookie_draft'   -- Subsequent seasons = rookie draft
    END AS acquired_via,
    dp.draft_id AS acquired_transaction_id,
    0 AS acquired_faab,
    lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr ON dp.draft_id = dr.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON dp.league_id = li.league_id
  WHERE dp.player_id IS NOT NULL  -- Filter out empty draft slots
),
trade_acquisitions_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    li.season,
    lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
    explode(transform(map_entries(t.adds), e -> 
      named_struct('player_id', e.key, 'to_roster_id', e.value)
    )) AS add_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type = 'trade' AND t.status = 'complete'  -- Only completed trades
),
trade_acquisitions AS (
  SELECT
    add_struct.player_id,
    league_id,
    season,
    add_struct.to_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS acquired_date,  -- Convert Sleeper epoch to timestamp
    leg AS acquired_week,
    'trade' AS acquired_via,
    transaction_id AS acquired_transaction_id,
    0 AS acquired_faab,  -- Trades don't cost FAAB
    cluster_key
  FROM trade_acquisitions_raw
),
waiver_acquisitions_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    t.type,
    t.waiver_bid,
    li.season,
    lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
    explode(transform(map_entries(t.adds), e -> 
      named_struct('player_id', e.key, 'to_roster_id', e.value)
    )) AS add_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type IN ('waiver', 'free_agent', 'commissioner')
    AND t.status = 'complete'  -- Only completed transactions (prevents phantom ownership from failed waivers)
    AND size(map_keys(t.adds)) > 0  -- Only process if there are adds
),
waiver_acquisitions AS (
  SELECT
    add_struct.player_id,
    league_id,
    season,
    add_struct.to_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS acquired_date,
    leg AS acquired_week,
    type AS acquired_via,  -- 'waiver' or 'free_agent'
    transaction_id AS acquired_transaction_id,
    coalesce(waiver_bid, 0) AS acquired_faab,  -- Free agents have NULL bid
    cluster_key
  FROM waiver_acquisitions_raw
),
-- Step 2: Extract all departure events with cluster_key
trade_departures_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    li.season,
    lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
    explode(transform(map_entries(t.drops), e -> 
      named_struct('player_id', e.key, 'from_roster_id', e.value)
    )) AS drop_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type = 'trade' AND t.status = 'complete'
),
trade_departures AS (
  SELECT
    drop_struct.player_id,
    league_id,
    season,
    drop_struct.from_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS departed_date,
    leg AS departed_week,
    'trade' AS departed_via,
    transaction_id AS departed_transaction_id,
    cluster_key
  FROM trade_departures_raw
),
drop_departures_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    li.season,
    lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
    explode(transform(map_entries(t.drops), e -> 
      named_struct('player_id', e.key, 'from_roster_id', e.value)
    )) AS drop_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type IN ('waiver', 'free_agent', 'commissioner')
    AND t.status = 'complete'  -- Only completed transactions
    AND size(map_keys(t.drops)) > 0  -- Only process if there are drops
),
drop_departures AS (
  SELECT
    drop_struct.player_id,
    league_id,
    season,
    drop_struct.from_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS departed_date,
    leg AS departed_week,
    'drop' AS departed_via,
    transaction_id AS departed_transaction_id,
    cluster_key
  FROM drop_departures_raw
),
-- Step 3: Combine all events
all_acquisitions AS (
  SELECT * FROM draft_acquisitions
  UNION ALL
  SELECT * FROM trade_acquisitions
  UNION ALL  
  SELECT * FROM waiver_acquisitions
),
all_departures AS (
  SELECT * FROM trade_departures
  UNION ALL
  SELECT * FROM drop_departures
),
-- Step 4: Match acquisitions to their next departure using cluster_key (enables cross-season matching)
ownership_periods AS (
  SELECT
    a.player_id,
    a.league_id,
    a.season,
    a.roster_id,
    a.acquired_date,
    a.acquired_week,
    a.acquired_via,
    a.acquired_transaction_id,
    a.acquired_faab,
    d.departed_date,
    d.departed_week,
    d.departed_via,
    d.departed_transaction_id,
    ROW_NUMBER() OVER (
      PARTITION BY a.player_id, a.league_id, a.roster_id, a.season, a.acquired_transaction_id
      ORDER BY coalesce(d.departed_date, CAST('2099-01-01' AS TIMESTAMP)) ASC
    ) AS rn
  FROM all_acquisitions a
  LEFT JOIN all_departures d
    ON a.player_id = d.player_id
    AND a.cluster_key = d.cluster_key  -- Match by cluster, not league_id (enables cross-season matching for dynasty)
    AND a.roster_id = d.roster_id
    AND coalesce(d.departed_date, CAST('2099-01-01' AS TIMESTAMP)) >   -- Departure must be after acquisition
        coalesce(a.acquired_date, CAST('1900-01-01' AS TIMESTAMP))     -- Drafts have NULL acquired_date, use old date for comparison
)
SELECT
  md5(concat(player_id, league_id, CAST(roster_id AS STRING), acquired_transaction_id)) AS ownership_id,
  player_id,
  league_id,
  season,
  roster_id,
  acquired_date,
  acquired_week,
  acquired_via,
  acquired_transaction_id,
  acquired_faab,
  departed_date,
  departed_week,
  departed_via,
  departed_transaction_id,
  CASE WHEN departed_date IS NULL THEN TRUE ELSE FALSE END AS is_current_roster,
  CASE 
    WHEN departed_week IS NOT NULL AND acquired_week IS NOT NULL
    THEN departed_week - acquired_week
    ELSE NULL  -- Cross-season ownership makes week calculation complex
  END AS weeks_owned
FROM ownership_periods
WHERE rn = 1;  -- Take only the earliest matching departure for each acquisition

In [0]:
-- =============================================================================
-- dim_draft_metadata: Draft configuration with calculated pick numbers
-- =============================================================================
-- PURPOSE:
--   Provides complete draft metadata including slot assignments and calculated
--   pick numbers for each roster/round combination. Enables draft analysis
--   without accessing raw draft tables.
--
-- GRAIN:
--   One row per (draft_id, roster_id, round)
--
-- COLUMNS:
--   draft_id:        Sleeper draft identifier
--   league_id:       Sleeper league identifier
--   season:          Season when draft occurred
--   roster_id:       Team identifier
--   slot:            Draft slot position (1-N)
--   round:           Draft round number
--   num_rounds:      Total rounds in this draft
--   num_teams:       Number of teams in draft
--   draft_type:      Draft format (snake or linear)
--   pick_no:         Calculated overall pick number
--   cluster_key:     League cluster for cross-season tracking
--   cluster_name:    Human-readable league name
--
-- BUSINESS LOGIC:
--   - Snake drafts: Odd rounds go 1,2,3..., even rounds go N,...,3,2,1
--   - Linear drafts: Same order every round
--   Example: 12-team snake, slot 3:
--     - Round 1 (odd): pick_no = 3
--     - Round 2 (even): pick_no = 22 (12*2 - 3 + 1)
--     - Round 3 (odd): pick_no = 27 (12*2 + 3)
--
-- DEPENDENCIES:
--   - sleeper_drafts_snapshot: Draft configuration
--   - sleeper_draft_slot_to_roster_snapshot: Slot assignments
--   - dim_league_clusters: League clustering
-- =============================================================================
CREATE OR REPLACE MATERIALIZED VIEW dim_draft_metadata AS
WITH draft_base AS (
  SELECT
    dr.draft_id,
    dr.league_id,
    dr.season,
    slot.slot,
    slot.roster_id,
    CAST(dr.settings['rounds'] AS INT) AS num_rounds,
    CAST(dr.settings['teams'] AS INT) AS num_teams,
    dr.type AS draft_type,
    lc.cluster_key,
    lc.cluster_name
  FROM workspace.sleeper_raw.sleeper_drafts_snapshot dr
  JOIN workspace.sleeper_raw.sleeper_draft_slot_to_roster_snapshot slot 
    ON dr.draft_id = slot.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li 
    ON dr.league_id = li.league_id
  JOIN dim_league_clusters lc
    ON lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) = lc.cluster_key
),
-- Generate all rounds for each draft/roster combination
rounds_expanded AS (
  SELECT
    draft_id,
    league_id,
    season,
    roster_id,
    slot,
    num_rounds,
    num_teams,
    draft_type,
    cluster_key,
    cluster_name,
    round
  FROM draft_base
  CROSS JOIN (
    SELECT pos AS round 
    FROM explode(sequence(1, (SELECT MAX(num_rounds) FROM draft_base))) AS t(pos)
  ) rounds
)
SELECT
  draft_id,
  league_id,
  season,
  roster_id,
  slot,
  round,
  num_rounds,
  num_teams,
  draft_type,
  -- Calculate pick number based on draft type
  CASE
    WHEN draft_type = 'snake' THEN
      CASE
        WHEN round % 2 = 1 THEN (round - 1) * num_teams + slot  -- Odd rounds: normal order
        ELSE round * num_teams - slot + 1                        -- Even rounds: reverse order
      END
    ELSE (round - 1) * num_teams + slot  -- Linear: same order every round
  END AS pick_no,
  cluster_key,
  cluster_name
FROM rounds_expanded;

In [0]:
-- =============================================================================
-- dim_draft_picks: Draft picks with manager and player information
-- =============================================================================
-- PURPOSE:
--   Complete draft pick information enriched with manager names and calculated
--   pick numbers. Enables draft analysis without accessing raw tables.
--
-- GRAIN:
--   One row per draft pick
--
-- COLUMNS:
--   draft_id:              Sleeper draft identifier
--   league_id:             Sleeper league identifier
--   season:                Season when draft occurred
--   player_id:             Sleeper player identifier (NULL for unpicked slots)
--   pick_no:               Overall pick number in draft
--   round:                 Draft round
--   roster_id:             Team that made the pick
--   cluster_key:           League cluster for cross-season tracking
--   cluster_name:          Human-readable league name
--   manager_display_name:  Manager who made the pick (with alias resolution)
--   draft_type:            Type of draft (startup_draft, rookie_draft)
--   pick_description:      Human-readable pick label
--
-- BUSINESS LOGIC:
--   - Identifies startup vs rookie drafts by checking if season = league first season
--   - Filters out NULL player_id slots (unpicked slots at end of drafts)
--   - Uses dim_manager_roster_map for canonical manager names
--
-- DEPENDENCIES:
--   - sleeper_draft_picks_snapshot: Raw draft pick data
--   - sleeper_drafts_snapshot: Draft metadata
--   - dim_manager_roster_map: Manager identity with aliases
--   - dim_league_clusters: League clustering
-- =============================================================================
CREATE OR REPLACE MATERIALIZED VIEW dim_draft_picks AS
SELECT
  dp.draft_id,
  dp.league_id,
  dr.season,
  dp.player_id,
  dp.pick_no,
  CAST(dp.round AS INT) AS round,
  dp.roster_id,
  lc.cluster_key,
  lc.cluster_name,
  mr.manager_display_name,
  CASE
    WHEN dr.season = lc.season_start THEN 'startup_draft'
    ELSE 'rookie_draft'
  END AS draft_type,
  CONCAT(dr.season, ' Round ', CAST(dp.round AS INT), ' Pick ', dp.pick_no) AS pick_description
FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr
  ON dp.draft_id = dr.draft_id
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li
  ON dp.league_id = li.league_id
JOIN dim_league_clusters lc
  ON lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) = lc.cluster_key
LEFT JOIN dim_manager_roster_map mr
  ON dp.league_id = mr.league_id
  AND dp.roster_id = mr.roster_id
  AND dr.season = mr.season
WHERE dp.player_id IS NOT NULL;